In [ ]:
# 1. Install the packages required for the experiment.
!pip install -q transformers==5.16.1 datasets==4.0.0 evaluate==0.4.6 accelerate==1.14.0 scikit-learn==1.6.1 pandas==2.2.3 numpy==2.1.3 matplotlib seaborn


In [ ]:
# 2. Import standard Python libraries.
import gc
import importlib.metadata as importlib_metadata
import json
import math
import random
from pathlib import Path


In [ ]:
# 3. Import machine-learning libraries.
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed


In [ ]:
# 4. Display the installed library versions.
packages = ("transformers", "datasets", "evaluate", "accelerate", "scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "torch")
for package in packages:
    print(f"{package}: {importlib_metadata.version(package)}")


In [ ]:
# 5. Check the available Colab hardware.
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 6. Configure project paths, model settings, and the five training seeds.
PROJECT_ROOT = Path("/content/diplomski")
DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results" / "repeated_experiments"
MODEL_NAME = "ProsusAI/finbert"
SEEDS = (43, 47, 53, 59, 61)
LABEL_IDS = [0, 1, 2]
LABEL_NAMES = ["bullish", "bearish", "neutral"]
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
SKIP_COMPLETED_RUNS = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FP16 = torch.cuda.is_available()

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Results root:", RESULTS_ROOT)
print("Seeds:", SEEDS)
print("Device:", DEVICE)


In [ ]:
# 7. Check that all random and chronological dataset files are available.
required_paths = [
    DATA_ROOT / split_name / f"{part}.csv"
    for split_name in ("random", "chronological")
    for part in ("train", "validation", "test")
]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing files:\n" + "\n".join(str(path) for path in missing_paths))
print("All six dataset files exist.")


In [ ]:
# 8. Load the random train, validation, and test CSV files.
random_frames = {
    part: pd.read_csv(DATA_ROOT / "random" / f"{part}.csv")
    for part in ("train", "validation", "test")
}
print("Random datasets loaded.")


In [ ]:
# 9. Inspect random dataset sizes, columns, and label distributions.
for part, frame in random_frames.items():
    print(f"Random {part}: {frame.shape}")
    required_columns = {"text", "market_direction", "labels"}
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"Missing columns in random/{part}.csv: {sorted(missing_columns)}")
print("\nRandom test labels:")
print(random_frames["test"]["labels"].value_counts().sort_index())


In [ ]:
# 10. Load the chronological train, validation, and test CSV files.
chronological_frames = {
    part: pd.read_csv(DATA_ROOT / "chronological" / f"{part}.csv")
    for part in ("train", "validation", "test")
}
print("Chronological datasets loaded.")


In [ ]:
# 11. Inspect chronological dataset sizes, dates, columns, and label distributions.
for part, frame in chronological_frames.items():
    required_columns = {"text", "market_direction", "labels", "timestamp"}
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"Missing columns in chronological/{part}.csv: {sorted(missing_columns)}")
    dates = pd.to_datetime(frame["timestamp"], utc=True)
    print(f"Chronological {part}: {frame.shape}; {dates.min()} -> {dates.max()}")
print("\nChronological test labels:")
print(chronological_frames["test"]["labels"].value_counts().sort_index())


In [ ]:
# 12. Load the FinBERT tokenizer and configure dynamic batch padding.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenizer loaded:", MODEL_NAME)


In [ ]:
# 13. Tokenize the random train, validation, and test datasets.
def tokenize_frame(frame):
    dataset = Dataset.from_pandas(frame[["text", "labels"]], preserve_index=False)
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=["text"],
    )

random_tokenized = {part: tokenize_frame(frame) for part, frame in random_frames.items()}
for part, dataset in random_tokenized.items():
    print(f"Random {part} tokenized: {len(dataset)} examples")


In [ ]:
# 14. Tokenize the chronological train, validation, and test datasets.
chronological_tokenized = {part: tokenize_frame(frame) for part, frame in chronological_frames.items()}
for part, dataset in chronological_tokenized.items():
    print(f"Chronological {part} tokenized: {len(dataset)} examples")


In [ ]:
# 15. Define the evaluation metric calculation.
def calculate_metrics(true_labels, predicted_labels):
    macro = precision_recall_fscore_support(
        true_labels, predicted_labels, labels=LABEL_IDS, average="macro", zero_division=0
    )
    weighted = precision_recall_fscore_support(
        true_labels, predicted_labels, labels=LABEL_IDS, average="weighted", zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(true_labels, predicted_labels)),
        "precision_macro": float(macro[0]),
        "recall_macro": float(macro[1]),
        "f1_macro": float(macro[2]),
        "weighted_f1": float(weighted[2]),
    }



In [ ]:
# 16. Define the original FinBERT baseline prediction procedure.
# FinBERT outputs textual sentiment (positive, negative, neutral), while the targets
# represent market direction (bullish, bearish, neutral); the ID mapping is technical,
# so baseline results and their McNemar comparison require cautious interpretation.
id2label = {0: "bullish", 1: "bearish", 2: "neutral"}
label2id = {"bullish": 0, "bearish": 1, "neutral": 2}

def predict_baseline(test_frame):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    model.to(DEVICE)
    model.eval()
    predictions = []
    texts = test_frame["text"].tolist()
    for start in range(0, len(texts), BATCH_SIZE):
        inputs = tokenizer(
            texts[start:start + BATCH_SIZE], return_tensors="pt", truncation=True,
            padding=True, max_length=MAX_LENGTH
        )
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
        with torch.no_grad():
            predictions.extend(model(**inputs).logits.argmax(dim=-1).cpu().tolist())
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return np.array(predictions)


In [ ]:
# 17. Run the original FinBERT baseline on the random test set.
random_baseline_predictions = predict_baseline(random_frames["test"])
random_true_labels = random_frames["test"]["labels"].to_numpy()
random_baseline_metrics = calculate_metrics(random_true_labels, random_baseline_predictions)
print("Random baseline metrics:", random_baseline_metrics)


In [ ]:
# 18. Save the original FinBERT random-split baseline metrics and predictions.
random_baseline_result = {
    "split": "random", "seed": "baseline", "model": "original_finbert",
    **random_baseline_metrics,
    "classification_report": classification_report(
        random_true_labels, random_baseline_predictions, labels=LABEL_IDS,
        target_names=LABEL_NAMES, output_dict=True, zero_division=0
    ),
    "confusion_matrix": confusion_matrix(
        random_true_labels, random_baseline_predictions, labels=LABEL_IDS
    ).tolist(),
}
random_results_dir = RESULTS_ROOT / "random"
random_results_dir.mkdir(parents=True, exist_ok=True)
with open(random_results_dir / "baseline_metrics.json", "w") as file:
    json.dump(random_baseline_result, file, indent=2)
pd.DataFrame({
    "true_label": random_true_labels,
    "baseline_prediction": random_baseline_predictions,
}).to_csv(random_results_dir / "baseline_predictions.csv", index=False)
print("Random baseline outputs saved.")


In [ ]:
# 19. Evaluate the original FinBERT on the chronological test set and save its outputs.
chronological_baseline_predictions = predict_baseline(chronological_frames["test"])
chronological_true_labels = chronological_frames["test"]["labels"].to_numpy()
chronological_baseline_metrics = calculate_metrics(chronological_true_labels, chronological_baseline_predictions)
chronological_baseline_result = {
    "split": "chronological", "seed": "baseline", "model": "original_finbert",
    **chronological_baseline_metrics,
    "classification_report": classification_report(
        chronological_true_labels, chronological_baseline_predictions, labels=LABEL_IDS,
        target_names=LABEL_NAMES, output_dict=True, zero_division=0
    ),
    "confusion_matrix": confusion_matrix(
        chronological_true_labels, chronological_baseline_predictions, labels=LABEL_IDS
    ).tolist(),
}
chronological_results_dir = RESULTS_ROOT / "chronological"
chronological_results_dir.mkdir(parents=True, exist_ok=True)
with open(chronological_results_dir / "baseline_metrics.json", "w") as file:
    json.dump(chronological_baseline_result, file, indent=2)
pd.DataFrame({
    "true_label": chronological_true_labels,
    "baseline_prediction": chronological_baseline_predictions,
}).to_csv(chronological_results_dir / "baseline_predictions.csv", index=False)
print("Chronological baseline metrics:", chronological_baseline_metrics)


In [ ]:
# 20. Prepare TF-IDF features for the random split.
random_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
random_x_train = random_vectorizer.fit_transform(random_frames["train"]["text"].fillna(""))
random_x_validation = random_vectorizer.transform(random_frames["validation"]["text"].fillna(""))
random_x_test = random_vectorizer.transform(random_frames["test"]["text"].fillna(""))
random_y_train = random_frames["train"]["labels"].to_numpy()
random_y_validation = random_frames["validation"]["labels"].to_numpy()
random_y_test = random_frames["test"]["labels"].to_numpy()
print("Random TF-IDF shapes:", random_x_train.shape, random_x_validation.shape, random_x_test.shape)


In [ ]:
# 21. Select the best logistic-regression C value on random validation data.
random_validation_scores = {}
for c_value in (0.1, 1.0, 10.0):
    candidate = LogisticRegression(C=c_value, max_iter=2000, random_state=43)
    candidate.fit(random_x_train, random_y_train)
    random_validation_scores[str(c_value)] = calculate_metrics(random_y_validation, candidate.predict(random_x_validation))["f1_macro"]
random_best_c = float(max(random_validation_scores, key=random_validation_scores.get))
print("Random validation macro F1 by C:", random_validation_scores)
print("Selected random C:", random_best_c)


In [ ]:
# 22. Train and evaluate the TF-IDF model on the random test set.
# The final model is intentionally trained only on the training split after C selection.
random_tfidf_model = LogisticRegression(C=random_best_c, max_iter=2000, random_state=43)
random_tfidf_model.fit(random_x_train, random_y_train)
random_tfidf_predictions = random_tfidf_model.predict(random_x_test)
random_tfidf_metrics = calculate_metrics(random_y_test, random_tfidf_predictions)
random_tfidf_result = {"split": "random", "seed": "not_applicable", "model": "tfidf_logistic_regression", "best_C": random_best_c, "validation_macro_f1_by_C": random_validation_scores, **random_tfidf_metrics, "classification_report": classification_report(random_y_test, random_tfidf_predictions, labels=LABEL_IDS, target_names=LABEL_NAMES, output_dict=True, zero_division=0), "confusion_matrix": confusion_matrix(random_y_test, random_tfidf_predictions, labels=LABEL_IDS).tolist()}
print("Random TF-IDF metrics:", random_tfidf_metrics)


In [ ]:
# 23. Save the random TF-IDF metrics and predictions.
with open(RESULTS_ROOT / "random" / "tfidf_metrics.json", "w") as file:
    json.dump(random_tfidf_result, file, indent=2)
pd.DataFrame({"true_label": random_y_test, "tfidf_prediction": random_tfidf_predictions}).to_csv(RESULTS_ROOT / "random" / "tfidf_predictions.csv", index=False)
print("Random TF-IDF outputs saved.")


In [ ]:
# 24. Prepare TF-IDF features for the chronological split.
chronological_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
chronological_x_train = chronological_vectorizer.fit_transform(chronological_frames["train"]["text"].fillna(""))
chronological_x_validation = chronological_vectorizer.transform(chronological_frames["validation"]["text"].fillna(""))
chronological_x_test = chronological_vectorizer.transform(chronological_frames["test"]["text"].fillna(""))
chronological_y_train = chronological_frames["train"]["labels"].to_numpy()
chronological_y_validation = chronological_frames["validation"]["labels"].to_numpy()
chronological_y_test = chronological_frames["test"]["labels"].to_numpy()
print("Chronological TF-IDF shapes:", chronological_x_train.shape, chronological_x_validation.shape, chronological_x_test.shape)


In [ ]:
# 25. Select the best logistic-regression C value on chronological validation data.
chronological_validation_scores = {}
for c_value in (0.1, 1.0, 10.0):
    candidate = LogisticRegression(C=c_value, max_iter=2000, random_state=43)
    candidate.fit(chronological_x_train, chronological_y_train)
    chronological_validation_scores[str(c_value)] = calculate_metrics(chronological_y_validation, candidate.predict(chronological_x_validation))["f1_macro"]
chronological_best_c = float(max(chronological_validation_scores, key=chronological_validation_scores.get))
print("Chronological validation macro F1 by C:", chronological_validation_scores)
print("Selected chronological C:", chronological_best_c)


In [ ]:
# 26. Train and evaluate the TF-IDF model on the chronological test set.
# The final model is intentionally trained only on the training split after C selection.
chronological_tfidf_model = LogisticRegression(C=chronological_best_c, max_iter=2000, random_state=43)
chronological_tfidf_model.fit(chronological_x_train, chronological_y_train)
chronological_tfidf_predictions = chronological_tfidf_model.predict(chronological_x_test)
chronological_tfidf_metrics = calculate_metrics(chronological_y_test, chronological_tfidf_predictions)
chronological_tfidf_result = {"split": "chronological", "seed": "not_applicable", "model": "tfidf_logistic_regression", "best_C": chronological_best_c, "validation_macro_f1_by_C": chronological_validation_scores, **chronological_tfidf_metrics, "classification_report": classification_report(chronological_y_test, chronological_tfidf_predictions, labels=LABEL_IDS, target_names=LABEL_NAMES, output_dict=True, zero_division=0), "confusion_matrix": confusion_matrix(chronological_y_test, chronological_tfidf_predictions, labels=LABEL_IDS).tolist()}
print("Chronological TF-IDF metrics:", chronological_tfidf_metrics)


In [ ]:
# 27. Save the chronological TF-IDF metrics and predictions.
with open(RESULTS_ROOT / "chronological" / "tfidf_metrics.json", "w") as file:
    json.dump(chronological_tfidf_result, file, indent=2)
pd.DataFrame({"true_label": chronological_y_test, "tfidf_prediction": chronological_tfidf_predictions}).to_csv(RESULTS_ROOT / "chronological" / "tfidf_predictions.csv", index=False)
print("Chronological TF-IDF outputs saved.")


In [ ]:
# 28. Define the exact McNemar p-value calculation.
def exact_mcnemar_p_value(first_only_correct, second_only_correct):
    discordant_total = first_only_correct + second_only_correct
    if discordant_total == 0:
        return 1.0
    smaller = min(first_only_correct, second_only_correct)
    lower_tail = sum(math.comb(discordant_total, i) for i in range(smaller + 1)) / (2 ** discordant_total)
    return min(1.0, 2 * lower_tail)


In [ ]:
# 29. Define the Trainer metric callback.
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    return calculate_metrics(labels, np.argmax(logits, axis=-1))


In [ ]:
# 30. Set all random generators for one training run.
def set_training_seed(seed):
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
# 31. Build the model and Trainer for one training run.
def create_trainer(split_name, seed, tokenized):
    run_dir = RESULTS_ROOT / split_name / f"seed_{seed}"
    checkpoint_dir = run_dir / "checkpoints"
    run_dir.mkdir(parents=True, exist_ok=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
    )
    model.to(DEVICE)
    args = TrainingArguments(
        output_dir=str(checkpoint_dir), learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS, weight_decay=0.01, eval_strategy="epoch",
        save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1_macro",
        greater_is_better=True, logging_strategy="epoch", report_to="none",
        seed=seed, data_seed=seed, fp16=FP16,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"], processing_class=tokenizer,
        data_collator=data_collator, compute_metrics=compute_metrics,
    )
    return trainer, model, run_dir


In [ ]:
# 32. Train the Trainer model.
def train_model(trainer):
    trainer.train()
    return trainer


In [ ]:
# 33. Generate test-set predictions from the trained model.
def predict_model(trainer, tokenized):
    return trainer.predict(tokenized["test"])


In [ ]:
# 34. Calculate fine-tuned metrics and prepare the result report.
def build_model_result(split_name, seed, prediction):
    true_labels = prediction.label_ids
    fine_predictions = prediction.predictions.argmax(axis=-1)
    metrics = calculate_metrics(true_labels, fine_predictions)
    result = {
        "split": split_name, "seed": seed, "model": "fine_tuned", **metrics,
        "classification_report": classification_report(
            true_labels, fine_predictions, labels=LABEL_IDS, target_names=LABEL_NAMES,
            output_dict=True, zero_division=0
        ),
        "confusion_matrix": confusion_matrix(
            true_labels, fine_predictions, labels=LABEL_IDS
        ).tolist(),
    }
    return result, true_labels, fine_predictions


In [ ]:
# 35. Calculate the McNemar comparison for one test set.
# This compares predictions on the same examples, but it does not remove the
# semantic difference between textual sentiment and market-direction labels.
def build_mcnemar_result(split_name, seed, true_labels, baseline_predictions, fine_predictions):
    original_only = int(np.sum((baseline_predictions == true_labels) & (fine_predictions != true_labels)))
    fine_tuned_only = int(np.sum((baseline_predictions != true_labels) & (fine_predictions == true_labels)))
    return {
        "split": split_name, "seed": seed,
        "original_only_correct": original_only,
        "fine_tuned_only_correct": fine_tuned_only,
        "exact_mcnemar_p_value": exact_mcnemar_p_value(original_only, fine_tuned_only),
    }


In [ ]:
# 36. Save one training run's metrics and predictions.
def save_run_outputs(run_dir, result, true_labels, baseline_predictions, fine_predictions):
    with open(run_dir / "metrics.json", "w") as file:
        json.dump(result, file, indent=2)
    pd.DataFrame({
        "true_label": true_labels,
        "baseline_prediction": baseline_predictions,
        "fine_tuned_prediction": fine_predictions,
    }).to_csv(run_dir / "predictions.csv", index=False)


In [ ]:
# 37. Clear GPU memory after one run.
def clear_training_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()


In [ ]:
# 38. Coordinate one complete fine-tuning run for a selected seed.
def train_one_seed(split_name, seed, tokenized, baseline_predictions):
    run_dir = RESULTS_ROOT / split_name / f"seed_{seed}"
    metrics_path = run_dir / "metrics.json"
    predictions_path = run_dir / "predictions.csv"
    if SKIP_COMPLETED_RUNS and metrics_path.exists() and predictions_path.exists():
        with open(metrics_path) as file:
            result = json.load(file)
        saved_predictions = pd.read_csv(predictions_path)
        true_labels = saved_predictions["true_label"].to_numpy()
        fine_predictions = saved_predictions["fine_tuned_prediction"].to_numpy()
        mcnemar = build_mcnemar_result(
            split_name, seed, true_labels, baseline_predictions, fine_predictions
        )
        print(f"Skipped completed run: {split_name}, seed {seed}")
        return result, mcnemar
    set_training_seed(seed)
    trainer, model, run_dir = create_trainer(split_name, seed, tokenized)
    trainer = train_model(trainer)
    prediction = predict_model(trainer, tokenized)
    result, true_labels, fine_predictions = build_model_result(split_name, seed, prediction)
    mcnemar = build_mcnemar_result(
        split_name, seed, true_labels, baseline_predictions, fine_predictions
    )
    save_run_outputs(run_dir, result, true_labels, baseline_predictions, fine_predictions)
    del trainer, model
    clear_training_memory()
    return result, mcnemar


In [ ]:
# 39. Train and evaluate the random split with seed 43.
if "summary_rows" not in globals():
    summary_rows = [
        {"split": "random", "seed": "baseline", "model": "original_finbert", **random_baseline_metrics},
        {"split": "chronological", "seed": "baseline", "model": "original_finbert", **chronological_baseline_metrics},
    ]
if "mcnemar_rows" not in globals():
    mcnemar_rows = []
result, mcnemar = train_one_seed("random", 43, random_tokenized, random_baseline_predictions)
summary_rows = [row for row in summary_rows if not (row["split"] == result["split"] and row["seed"] == result["seed"] and row["model"] == result["model"])]
mcnemar_rows = [row for row in mcnemar_rows if not (row["split"] == mcnemar["split"] and row["seed"] == mcnemar["seed"])]
summary_rows.append({key: result[key] for key in ("split", "seed", "model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "weighted_f1")})
mcnemar_rows.append(mcnemar)
print("random", 43, "macro F1:", result["f1_macro"], "McNemar p:", mcnemar["exact_mcnemar_p_value"])


In [ ]:
# 40. Train and evaluate the random split with seed 47.
result, mcnemar = train_one_seed("random", 47, random_tokenized, random_baseline_predictions)
summary_rows = [row for row in summary_rows if not (row["split"] == result["split"] and row["seed"] == result["seed"] and row["model"] == result["model"])]
mcnemar_rows = [row for row in mcnemar_rows if not (row["split"] == mcnemar["split"] and row["seed"] == mcnemar["seed"])]
summary_rows.append({key: result[key] for key in ("split", "seed", "model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "weighted_f1")})
mcnemar_rows.append(mcnemar)
print(result["split"], result["seed"], result["f1_macro"], mcnemar["exact_mcnemar_p_value"])


In [ ]:
# 41. Train and evaluate the random split with seed 53.
result, mcnemar = train_one_seed("random", 53, random_tokenized, random_baseline_predictions)
summary_rows = [row for row in summary_rows if not (row["split"] == result["split"] and row["seed"] == result["seed"] and row["model"] == result["model"])]
mcnemar_rows = [row for row in mcnemar_rows if not (row["split"] == mcnemar["split"] and row["seed"] == mcnemar["seed"])]
summary_rows.append({key: result[key] for key in ("split", "seed", "model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "weighted_f1")})
mcnemar_rows.append(mcnemar)
print(result["split"], result["seed"], result["f1_macro"], mcnemar["exact_mcnemar_p_value"])


In [ ]:
# 42. Train and evaluate the random split with seed 59.
result, mcnemar = train_one_seed("random", 59, random_tokenized, random_baseline_predictions)
summary_rows = [row for row in summary_rows if not (row["split"] == result["split"] and row["seed"] == result["seed"] and row["model"] == result["model"])]
mcnemar_rows = [row for row in mcnemar_rows if not (row["split"] == mcnemar["split"] and row["seed"] == mcnemar["seed"])]
summary_rows.append({key: result[key] for key in ("split", "seed", "model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "weighted_f1")})
mcnemar_rows.append(mcnemar)
print(result["split"], result["seed"], result["f1_macro"], mcnemar["exact_mcnemar_p_value"])


In [ ]:
# 43. Train and evaluate the random split with seed 61.
result, mcnemar = train_one_seed("random", 61, random_tokenized, random_baseline_predictions)
summary_rows = [row for row in summary_rows if not (row["split"] == result["split"] and row["seed"] == result["seed"] and row["model"] == result["model"])]
mcnemar_rows = [row for row in mcnemar_rows if not (row["split"] == mcnemar["split"] and row["seed"] == mcnemar["seed"])]
summary_rows.append({key: result[key] for key in ("split", "seed", "model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "weighted_f1")})
mcnemar_rows.append(mcnemar)
print(result["split"], result["seed"], result["f1_macro"], mcnemar["exact_mcnemar_p_value"])


In [ ]:
# 44. Train and evaluate the chronological split with seed 43.
result, mcnemar = train_one_seed("chronological", 43, chronological_tokenized, chronological_baseline_predictions)
summary_rows = [row for row in summary_rows if not (row["split"] == result["split"] and row["seed"] == result["seed"] and row["model"] == result["model"])]
mcnemar_rows = [row for row in mcnemar_rows if not (row["split"] == mcnemar["split"] and row["seed"] == mcnemar["seed"])]
summary_rows.append({key: result[key] for key in ("split", "seed", "model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "weighted_f1")})
mcnemar_rows.append(mcnemar)
print(result["split"], result["seed"], result["f1_macro"], mcnemar["exact_mcnemar_p_value"])


In [ ]:
# 45. Train and evaluate the chronological split with seed 47.
result, mcnemar = train_one_seed("chronological", 47, chronological_tokenized, chronological_baseline_predictions)
summary_rows = [row for row in summary_rows if not (row["split"] == result["split"] and row["seed"] == result["seed"] and row["model"] == result["model"])]
mcnemar_rows = [row for row in mcnemar_rows if not (row["split"] == mcnemar["split"] and row["seed"] == mcnemar["seed"])]
summary_rows.append({key: result[key] for key in ("split", "seed", "model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "weighted_f1")})
mcnemar_rows.append(mcnemar)
print(result["split"], result["seed"], result["f1_macro"], mcnemar["exact_mcnemar_p_value"])


In [ ]:
# 46. Train and evaluate the chronological split with seed 53.
result, mcnemar = train_one_seed("chronological", 53, chronological_tokenized, chronological_baseline_predictions)
summary_rows = [row for row in summary_rows if not (row["split"] == result["split"] and row["seed"] == result["seed"] and row["model"] == result["model"])]
mcnemar_rows = [row for row in mcnemar_rows if not (row["split"] == mcnemar["split"] and row["seed"] == mcnemar["seed"])]
summary_rows.append({key: result[key] for key in ("split", "seed", "model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "weighted_f1")})
mcnemar_rows.append(mcnemar)
print(result["split"], result["seed"], result["f1_macro"], mcnemar["exact_mcnemar_p_value"])


In [ ]:
# 47. Train and evaluate the chronological split with seed 59.
result, mcnemar = train_one_seed("chronological", 59, chronological_tokenized, chronological_baseline_predictions)
summary_rows = [row for row in summary_rows if not (row["split"] == result["split"] and row["seed"] == result["seed"] and row["model"] == result["model"])]
mcnemar_rows = [row for row in mcnemar_rows if not (row["split"] == mcnemar["split"] and row["seed"] == mcnemar["seed"])]
summary_rows.append({key: result[key] for key in ("split", "seed", "model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "weighted_f1")})
mcnemar_rows.append(mcnemar)
print(result["split"], result["seed"], result["f1_macro"], mcnemar["exact_mcnemar_p_value"])


In [ ]:
# 48. Train and evaluate the chronological split with seed 61.
result, mcnemar = train_one_seed("chronological", 61, chronological_tokenized, chronological_baseline_predictions)
summary_rows = [row for row in summary_rows if not (row["split"] == result["split"] and row["seed"] == result["seed"] and row["model"] == result["model"])]
mcnemar_rows = [row for row in mcnemar_rows if not (row["split"] == mcnemar["split"] and row["seed"] == mcnemar["seed"])]
summary_rows.append({key: result[key] for key in ("split", "seed", "model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "weighted_f1")})
mcnemar_rows.append(mcnemar)
print(result["split"], result["seed"], result["f1_macro"], mcnemar["exact_mcnemar_p_value"])


In [ ]:
# 49. Aggregate all metrics and save the complete experiment tables.
summary = pd.DataFrame(summary_rows)
mcnemar_results = pd.DataFrame(mcnemar_rows)
fine_tuned_mean_std = (
    summary[summary["model"] == "fine_tuned"]
    .groupby("split")[["accuracy", "precision_macro", "recall_macro", "f1_macro", "weighted_f1"]]
    .agg(["mean", "std"])
)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
summary.to_csv(RESULTS_ROOT / "metrics_all_runs.csv", index=False)
mcnemar_results.to_csv(RESULTS_ROOT / "mcnemar_all_runs.csv", index=False)
fine_tuned_mean_std.to_csv(RESULTS_ROOT / "fine_tuned_mean_std.csv")
print("Metrics saved.")
display(summary)
display(mcnemar_results)
display(fine_tuned_mean_std)


In [ ]:
# 50. Verify that every expected experiment output has been saved.
expected = [
    RESULTS_ROOT / "metrics_all_runs.csv",
    RESULTS_ROOT / "mcnemar_all_runs.csv",
    RESULTS_ROOT / "fine_tuned_mean_std.csv",
]
for split_name in ("random", "chronological"):
    expected.extend([
        RESULTS_ROOT / split_name / "baseline_metrics.json",
        RESULTS_ROOT / split_name / "baseline_predictions.csv",
        RESULTS_ROOT / split_name / "tfidf_metrics.json",
        RESULTS_ROOT / split_name / "tfidf_predictions.csv",
    ])
    for seed in SEEDS:
        expected.extend([
            RESULTS_ROOT / split_name / f"seed_{seed}" / "metrics.json",
            RESULTS_ROOT / split_name / f"seed_{seed}" / "predictions.csv",
        ])
missing = [path for path in expected if not path.exists()]
if missing:
    raise FileNotFoundError("Missing outputs:\n" + "\n".join(str(path) for path in missing))
print(f"Verified {len(expected)} output files.")
